# Electromagnetism

This notebook covers **5 electromagnetic equations**, from electrostatics to superconductivity:

1. **Electrostatic Poisson Equation** — potential from charge distributions
2. **Electromagnetic Wave Equation (1D)** — EM wave propagation from Maxwell's equations
3. **Telegraph Equation** — signal propagation on lossy transmission lines
4. **Skin Effect Equation** — field penetration depth in conductors
5. **London Equations** — magnetic field expulsion in superconductors

Each section explores the **physical interpretation** including field penetration depth, wave propagation characteristics, and signal attenuation.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from diff_eq_solver import registry
from diff_eq_solver.visualizer import plot_ode_solution, plot_phase_portrait, plot_pde_heatmap, plot_pde_snapshots, plot_3d_surface, plot_comparison, plot_orbit, plot_potential_and_wavefunction
from IPython.display import Math, display

## 1. Electrostatic Poisson Equation

The **Poisson equation** for electrostatic potential:

$$\nabla^2 \varphi = -\frac{\rho}{\varepsilon_0}$$

**Physical context:** Given a charge distribution $\rho(x,y)$, this equation determines the electric potential $\varphi$ everywhere. For a point charge in 2D, the fundamental solution (Green's function) is $\varphi = \frac{1}{2\pi}\ln(1/r)$. The electric field is then $\mathbf{E} = -\nabla\varphi$.

In [ ]:
# Symbolic solve — 2D Green's function for a point charge
es_poisson = registry.get('electrostatic_poisson')
sym_sol = es_poisson.symbolic_solve()
print('Symbolic solution:')
if sym_sol.expression is not None:
    display(Math(r'\varphi(x,y) = ' + str(sym_sol.expression)))
print('\nDescription:', sym_sol.description if hasattr(sym_sol, 'description') else sym_sol.info)

In [ ]:
# Numerical solve — Gauss-Seidel on a 2D grid with point charge at center
num_sol = es_poisson.numerical_solve(Nx=64, Ny=64, max_iter=5000, tol=1e-6)
data = num_sol.data
phi = data['phi']
x = data['x']
y = data['y']
info = data['info']
print(f'Converged in {info["iterations"]} iterations, residual={info["final_residual"]:.2e}')

fig, ax = plt.subplots(figsize=(8, 6))
X, Y = np.meshgrid(x, y)
contour = ax.contourf(X, Y, phi, levels=30, cmap='RdYlBu_r')
fig.colorbar(contour, ax=ax, label='$\\varphi$ (potential)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Electrostatic Poisson: Point Charge at Center')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 2. Electromagnetic Wave Equation (1D)

Derived from **Maxwell's curl equations** in free space:

$$\frac{\partial^2 E}{\partial t^2} = c^2 \frac{\partial^2 E}{\partial x^2}$$

**Physical context:** This describes electromagnetic wave propagation at the speed of light $c = 1/\sqrt{\mu_0\varepsilon_0}$. A Gaussian pulse initial condition splits into two counter-propagating pulses. In dimensionless units ($c=1$), the d'Alembert solution gives $E(x,t) = \frac{1}{2}[f(x-t) + g(x+t)]$.

In [ ]:
# Symbolic solve — d'Alembert solution for Gaussian pulse
em_wave = registry.get('electromagnetic_wave_1d')
sym_sol = em_wave.symbolic_solve()
print('Symbolic solution (dAlembert):')
if sym_sol.expression is not None:
    display(Math(r'E(x,t) = ' + str(sym_sol.expression)))
print('\nDescription:', sym_sol.description if hasattr(sym_sol, 'description') else '')

In [ ]:
# Numerical solve — FDTD-like scheme
num_sol = em_wave.numerical_solve(Nx=200, Nt=500, L=1.0, T=1.0)
data = num_sol.data
x = data['x']
snapshots = data['snapshots']
times = data['times']
print(f'Courant number: {data["info"]["courant_number"]:.4f}')
print(f'Snapshots: {len(snapshots)} frames')

# Build u matrix from snapshots
u_matrix = np.array(snapshots)
t_arr = np.array(times)

fig1 = plot_pde_heatmap(x, t_arr, u_matrix, title='EM Wave Propagation: $E_{tt} = c^2 E_{xx}$')
plt.show()

# Plot selected snapshots showing pulse splitting
fig2 = plot_pde_snapshots(x, u_matrix, list(range(len(snapshots))), title='EM Wave: Gaussian Pulse Splitting')
plt.show()

## 3. Telegraph Equation

The **telegraph equation** for voltage/current on a lossy transmission line:

$$\frac{\partial^2 u}{\partial x^2} = LC\,\frac{\partial^2 u}{\partial t^2} + (RC+GL)\,\frac{\partial u}{\partial t} + RG\,u$$

**Physical context:** Models signal propagation along cables (e.g., transatlantic telegraph cables). The parameters $R, L, G, C$ are per-unit-length resistance, inductance, conductance, and capacitance. Key features:
- **Phase velocity:** $v = 1/\sqrt{LC}$
- **Attenuation:** $\alpha = (RC + GL)/(2\sqrt{LC})$
- **Signal distortion:** different frequency components attenuate at different rates

In [ ]:
# Symbolic solve — attenuated travelling wave
telegraph = registry.get('telegraph_equation')
sym_sol = telegraph.symbolic_solve()
print('Symbolic solution:')
if sym_sol.expression is not None:
    display(Math(r'u(x,t) \sim ' + str(sym_sol.expression)))
print('\nDescription:', sym_sol.description if hasattr(sym_sol, 'description') else '')

In [ ]:
# Numerical solve — explicit finite difference
num_sol = telegraph.numerical_solve(R=0.1, L=1.0, G=0.01, C=1.0, Nx=200, Nt=1000, domain_length=2.0, T=3.0)
data = num_sol.data
x = data['x']
snapshots = data['snapshots']
times = data['times']
info = data['info']
print(f'Phase velocity v = {1.0/np.sqrt(info["L"]*info["C"]):.4f}')
print(f'Attenuation alpha = {(info["R"]*info["C"]+info["G"]*info["L"])/(2*np.sqrt(info["L"]*info["C"])):.4f}')
print(f'Courant number: {info["courant_number"]:.4f}')

u_matrix = np.array(snapshots)
t_arr = np.array(times)

fig1 = plot_pde_heatmap(x, t_arr, u_matrix, title='Telegraph Equation: Signal with Attenuation')
plt.show()

# Show signal attenuation over time at a fixed point
fig, ax = plt.subplots(figsize=(10, 5))
monitor_idx = len(x) // 2
peak_vals = [s[monitor_idx] for s in snapshots]
ax.plot(times, peak_vals, 'ro-', markersize=4)
ax.set_xlabel('Time')
ax.set_ylabel('u(x_mid, t)')
ax.set_title('Telegraph Equation: Signal Attenuation at Midpoint')
ax.grid(True)
plt.tight_layout()
plt.show()

## 4. Skin Effect Equation

The **skin effect** governs field penetration into a conductor:

$$\frac{\partial^2 E}{\partial z^2} = \mu\sigma\,\frac{\partial E}{\partial t}$$

**Physical context:** At high frequencies, electromagnetic fields penetrate only a thin layer (the **skin depth**) into a conductor:

$$\delta = \sqrt{\frac{2}{\mu\sigma\omega}}$$

The field decays as $E(z,t) = E_0 e^{-z/\delta}\cos(\omega t - z/\delta)$. This has profound implications for:
- AC resistance (increases with frequency)
- Shielding effectiveness
- Eddy current losses in transformers

In [ ]:
# Symbolic solve — analytic time-harmonic solution
skin = registry.get('skin_effect_equation')
sym_sol = skin.symbolic_solve(mu=1.0, sigma=1.0, omega=1.0, E0=1.0)
print('Symbolic solution:')
if sym_sol.expression is not None:
    display(Math(r'E(z,t) = ' + str(sym_sol.expression)))
delta_sym = sym_sol.parameters.get('delta', None)
if delta_sym is not None:
    print(f'\nSkin depth delta = {delta_sym}')
print('\nDescription:', sym_sol.description if hasattr(sym_sol, 'description') else '')

In [ ]:
# Numerical solve — Crank-Nicolson
num_sol = skin.numerical_solve(mu=1.0, sigma=1.0, omega=1.0, E0=1.0, Nz=100, Nt=500, Lz=5.0, T=10.0)
data = num_sol.data
z = data['z']
snapshots = data['snapshots']
times = data['times']
info = data['info']
delta = info['skin_depth']
print(f'Skin depth delta = {delta:.4f}')
print(f'Crank-Nicolson parameter r = {info["r_cn"]:.4f}')

# Plot field profile at several times showing exponential decay with depth
fig, ax = plt.subplots(figsize=(10, 6))
n_plot = min(6, len(snapshots))
indices = np.linspace(0, len(snapshots) - 1, n_plot, dtype=int)
colors = plt.cm.viridis(np.linspace(0, 1, n_plot))
for i, idx in enumerate(indices):
    ax.plot(z, snapshots[idx], color=colors[i], label=f't = {times[idx]:.2f}')
# Overlay analytic envelope
z_fine = np.linspace(0, z[-1], 200)
envelope = np.exp(-z_fine / delta)
ax.plot(z_fine, envelope, 'k--', linewidth=2, label=f'Envelope $e^{{-z/\\delta}}$, $\\delta$={delta:.2f}')
ax.plot(z_fine, -envelope, 'k--', linewidth=2)
ax.set_xlabel('Depth z (skin depths)')
ax.set_ylabel('E(z, t)')
ax.set_title('Skin Effect: Field Penetration into Conductor')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 5. London Equations (Penetration Depth)

The **London equations** describe magnetic field expulsion in superconductors:

$$\frac{d^2 B}{dz^2} = \frac{B}{\lambda_L^2}$$

**Physical context:** When a magnetic field is applied to a superconductor, it decays exponentially from the surface with the **London penetration depth** $\lambda_L$:

$$B(z) = B_0 \exp(-z / \lambda_L)$$

This is the macroscopic manifestation of the **Meissner effect** — the complete expulsion of magnetic fields from the interior of a superconductor. Typical values: $\lambda_L \approx 20$-$200$ nm for metallic superconductors.

In [ ]:
# Symbolic solve — physical solution
london = registry.get('london_equations')
sym_sol = london.symbolic_solve(lambda_L=1.0, B0=1.0)
print('Symbolic solution:')
if sym_sol.expression is not None:
    display(Math(r'B(z) = ' + str(sym_sol.expression)))
print('\nDescription:', sym_sol.description if hasattr(sym_sol, 'description') else '')

In [ ]:
# Numerical solve — ODE integration
num_sol = london.numerical_solve(lambda_L=1.0, B0=1.0, z_max=5.0, Nz=200)
data = num_sol.data
z_vals = data['z']
B_vals = data['B']
B_analytic = data['B_analytic']
info = data['info']
print(f'London penetration depth: lambda_L = {info["lambda_L"]:.4f}')
print(f'Max error vs analytic: {info["max_error_vs_analytic"]:.2e}')

# Plot numerical vs analytic with penetration depth marker
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z_vals, B_vals, 'b-', linewidth=2, label='Numerical')
ax.plot(z_vals, B_analytic, 'r--', linewidth=2, label='Analytic: $B_0 e^{-z/\\lambda_L}$')
# Mark the penetration depth
ax.axvline(x=info['lambda_L'], color='green', linestyle=':', linewidth=2, label=f'$\\lambda_L$ = {info["lambda_L"]:.1f}')
ax.axhline(y=np.exp(-1), color='gray', linestyle=':', alpha=0.5, label='$B_0/e$ (1/e decay)')
ax.set_xlabel('Depth z (units of $\\lambda_L$)')
ax.set_ylabel('B(z) / $B_0$')
ax.set_title('London Penetration Depth: $B(z) = B_0 \\exp(-z/\\lambda_L)$')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()